# Forms and Input Validation

CSC-239 · Module 12 · Lesson 4 of 4

You can connect a button to a model and refresh a label. Now you will read editable text, decide whether it is acceptable, and preserve the last valid value when a new request fails.

Use a fresh **Java** kernel and run the supplied setup. Open the Workspace **Desktop** view for native interaction. Run each complete example from its beginning before a new test case. Check initial notebook output and later visible results separately.


## Learning Goals

- Build a labeled JavaFX form that validates input before changing its model.
- Test valid, boundary, malformed, and repeated submissions with pointer and keyboard input.


## Why This Matters

An equipment request must contain a whole-number quantity from 1 through 5. A useful form explains how to correct an invalid request and keeps the last successfully saved quantity.


## Check Your Starting Point

Explain Integer.parseInt, trim, NumberFormatException, and a catch block. Recall why a candidate local variable can be checked before assigning a private field. Describe event registration, model state, and the label refresh from the previous lesson.

**My explanation:**


## Concept

### Read editable text at the right time

A **text input control** lets a user enter and edit text. JavaFX's TextField is a single-line input control. `new TextField("2")` starts it with the text 2. The field still contains a String; typing digits does not turn that text into an int.

A **form submission** is a deliberate action that reads current field text and attempts to apply it. Our Save quantity button requests submission. Creating the field does not save its starting text, and typing a new value does not automatically change the saved quantity.

Read `input.getText()` inside the submit handler. Reading it once during construction would retain the initial text instead of the user's later edit. Keep the field text and the saved model value distinct: one is a proposed request, while the other records the last accepted request.

### Check a candidate before replacing state

**Validate before mutation** means checking a proposed value before replacing valid model state. Mutation means changing stored state. The QuantityModel starts with quantity zero to represent nothing saved. Zero is its initial marker; it is not an accepted submitted quantity.

The save method follows this sequence:

1. Trim surrounding whitespace and attempt to parse the result as an int.
2. Store that parsed value in the local candidate variable.
3. Reject a candidate below 1 or above 5 without assigning the quantity field.
4. Assign quantity only after both checks succeed.
5. Return a message describing the result.

If parsing fails, the catch returns a correction message. Blank text, a word, and an integer too large for int all fail parsing. A parsed zero or six reaches the range check instead. This distinction produces useful feedback without treating every failure as the same problem.

This complete model check can run in an ordinary Java cell; it does not create a window:

```java
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
QuantityModel model = new QuantityModel();
System.out.println(model.save("2"));
System.out.println(model.save("six"));
System.out.println("Still saved: " + model.getQuantity());
System.out.println(model.save(" 5 "));
System.out.println("Now saved: " + model.getQuantity());
```

It prints Saved: 2, Enter a whole number from 1 to 5., Still saved: 2, Saved: 5, and Now saved: 5 on separate lines. The rejected word does not overwrite the earlier two. Trimming allows the surrounding spaces in the final request.

These field submissions are non-null Strings. A general-purpose method used by other callers would need a separate policy for null; this model's caller requirement is non-null text.

### Give useful visible feedback

**Visible validation feedback** is a text message that explains success or tells the user how to correct an invalid input. A Label initially says Nothing saved. After submission, `status.setText(model.save(input.getText()))` displays the model's response.

The feedback names the allowed range. It does not rely on red or green to distinguish outcomes. Wrapping lets the correction message occupy more than one line when needed. Test the full sentence in the actual window, including the end of the message.

An error message replaces the previous success message in the view, but it does not replace the previously saved value in the model. Those are different changes. A screenshot of the error alone cannot prove that saved state was preserved. Test the model getter separately, as in the complete check above.

### Associate the visible label with its input

**Control label association** connects a visible Label to its intended input control. `fieldLabel.setLabelFor(input)` states that the Quantity (1 to 5) label describes this TextField.

Keep that label visible even after the user types. A label association gives the interface an explicit relationship; it does not prove that every accessibility feature or every input method works. Also inspect the visible wording and move through the controls with the keyboard.

### Share one response across input methods

**Shared keyboard and pointer submission** uses the same handler for a button action and Enter in a text field. EventHandler is the JavaFX interface for event-handling behavior. Its type argument ActionEvent identifies the notification this handler accepts.

The worked example creates one local variable, `submit`, with type `EventHandler<ActionEvent>`. Its lambda reads the current field, asks the model to save, and updates status. `save.setOnAction(submit)` registers it on the button; `input.setOnAction(submit)` registers the same behavior for Enter in the field.

Registering the same handler on both controls prevents the two paths from acquiring different validation rules. Pointer activation of Save quantity and Enter while editing must agree about the accepted range and messages. Tab can move between controls, and Space can activate the focused button.

All these callbacks are short and run on the JavaFX Application Thread. They perform parsing and model updates without waiting for other work. The earlier supplied Fx support constructs the interface on that same thread.

### Test a sequence, not just one valid entry

Begin with a fresh model and window. Confirm Nothing saved. even though the field starts with 2. Save that value, then submit a word, zero, and six. Confirm the appropriate correction text each time. Use Enter to submit a valid boundary value and confirm recovery.

For the ordinary model, call getQuantity after every rejection and compare it with the most recent accepted value. Test both endpoints, surrounding spaces, blank input, and an oversized integer. A final successful request demonstrates that an earlier failure did not make the form unusable.

The notebook's initial Saved quantity: 0 line is a construction report. Later clicks update the model and visible status; they do not rewrite that earlier console line. Record later observations in your test table rather than treating the initial line as live state.


### Prepare this kernel

This is supplied course support for running JavaFX inside IJava. Run it once after starting or restarting this notebook's Java kernel. The message `FX ready` means the support has initialized JavaFX and completed an operation on its application thread. Open the Workspace **Desktop** view to see the windows created by later cells.

`Fx.run(() -> { ... })` performs the enclosed UI work on the JavaFX Application Thread and waits for that short operation to finish. Use it for reading as well as changing a live window or its controls. `Fx.closeWindows()` hides the windows created by this kernel before another example opens its own. `Fx.start()` is safe to call again; it keeps JavaFX available after the last window closes.

The implementation below is provided runtime support. You do not need to write its thread-coordination machinery for this lesson. A thread is one sequence of execution; Module 13 studies how to coordinate more than one. Here your responsibility is to use the documented support operations and keep UI work short. The support uses a completion signal, a time limit, and an error holder so a later cell does not silently continue after unfinished or failed UI work.

Do not call `Platform.exit()` during notebook practice. That ends the toolkit for this kernel; restart the kernel and rerun setup if you do so. Closing a window is different from ending the toolkit. If setup reports a display error, check that the Workspace Desktop is running, then restart the kernel and rerun setup. A JavaFX window appears in the Desktop, not as an inline notebook control.


In [ ]:
import javafx.application.Platform;
import javafx.stage.Window;
import java.util.ArrayList;
import java.util.concurrent.CountDownLatch;
import java.util.concurrent.TimeUnit;
import java.util.concurrent.atomic.AtomicReference;
class Fx {
    static void run(Runnable action) throws InterruptedException {
        if (Platform.isFxApplicationThread()) {
            action.run();
            return;
        }
        CountDownLatch done = new CountDownLatch(1);
        AtomicReference<Throwable> failure = new AtomicReference<Throwable>();
        Platform.runLater(() -> {
            try { action.run(); }
            catch (Throwable error) { failure.set(error); }
            finally { done.countDown(); }
        });
        if (!done.await(10, TimeUnit.SECONDS)) {
            throw new IllegalStateException("FX operation timed out; restart the kernel.");
        }
        if (failure.get() != null) { throw new RuntimeException(failure.get()); }
    }
    static void start() throws InterruptedException {
        try { Platform.startup(() -> Platform.setImplicitExit(false)); }
        catch (IllegalStateException alreadyStarted) {
            // This call is also safe when this kernel already started JavaFX.
        }
        run(() -> Platform.setImplicitExit(false));
    }
    static void closeWindows() throws InterruptedException {
        run(() -> {
            for (Window window : new ArrayList<Window>(Window.getWindows())) {
                window.hide();
            }
        });
    }
}
Fx.start();
System.out.println("FX ready");


## Video Demonstration

Watch a valid submission, a rejected edit, and recovery through Enter in the field. Predict the feedback and the saved model value separately. Compare the visible result with the model checks in this notebook.

<video controls preload="metadata" width="960">
  <source src="media/04_forms_and_input_validation/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/04_forms_and_input_validation/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the forms and input validation demonstration transcript](media/04_forms_and_input_validation/transcript.md).


## Worked Example

**Subgoal 1: build a labeled form.** Create the TextField, its associated label, Save quantity, and wrapping status text.

**Subgoal 2: validate the candidate.** Keep accepted quantity unchanged until parsing and range checks succeed.

**Subgoal 3: share submission.** Register one handler on the button and input, then test both input methods.


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("2");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Quantity Form");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});


Expected output:

```text
Saved quantity: 0
```

The initial quantity is zero and the status says Nothing saved. Activating Save quantity with the initial text 2 displays Saved: 2. A word produces the whole-number message; zero or six produces the range message. Each rejection leaves the saved two unchanged. Enter with 5 displays Saved: 5 and updates the saved value.


## Predict, Run, Trace, and Explain

### Predict a new form sequence

Read the complete Supply Request program before running it. Predict the initial notebook output, field text and status separately. Then predict the status and saved quantity after submitting 4, then 0, then a word, and finally the padded text ` 3 `. Explain whether typing alone changes the saved model. Keep your original prediction; do not open the answer yet.

My prediction:

My actual field text and status observations:

My model getter evidence:

My explanation using the relevant source:

What I changed after comparing the evidence:


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});


Run the supplied setup if this Java kernel has just started, then run the complete program. In the Workspace Desktop, inspect the initial field and status. Without rerunning the program between submissions, replace the field text and submit these requests in order: `4`, `0`, `6`, empty text, `six`, `2147483648`, ` 3 `, `1`, `5`, `word`, and `2`. Use a button action for requests 1, 3, 5, 7, 9 and 11; use Enter while the field has focus for requests 2, 4, 6, 8 and 10. Use pointer clicks for some button actions and Tab plus Space for at least one. Select the current field text with Ctrl+A before replacing it; for empty text, press Backspace after selecting it. Preserve the spaces around ` 3 `. Record the full status after each submission and compare it with your original prediction. Then close the native window with Alt+F4, rerun the complete program in the same kernel, check its initial field and status again, and close the recreated window. Explain what changed and which evidence reports saved state.

My prediction:

My actual field text and status observations:

My model getter evidence:

My explanation using the relevant source:

What I changed after comparing the evidence:

### Trace candidate, saved value and feedback

Trace the first valid submission followed by 0, blank text and an oversized integer. For each, identify the current field String, whether parsing succeeds, whether the range check returns, whether quantity is assigned, and which message reaches the status Label. Explain why a correction message can appear while the previous saved value remains. Identify the label association and the two registrations of the same submit handler. After writing your trace, run the complete ordinary model check to compare getter values; keep that evidence separate from the native status observations. Write a post-run explanation using your own words.

My prediction:

My actual field text and status observations:

My model getter evidence:

My explanation using the relevant source:

What I changed after comparing the evidence:

<details>
<summary>Show answer</summary>

The new window is Supply Request, the field contains 4, and status begins as Nothing saved. The construction output is Saved quantity: 0. Creating a TextField does not submit its text. The first submission displays Saved: 4. Parsed 0 and 6 receive Use a quantity from 1 to 5.; empty text, six and 2147483648 receive Enter a whole number from 1 to 5. Those rejections do not assign quantity. The separate getter check verifies that four remains stored. The padded request ` 3 ` is trimmed and accepted, then the two endpoints 1 and 5 are accepted. After word is rejected, the final 2 is accepted. Both controls use the same submit object, so their submission rules agree. The candidate is local: parsing and range checks happen before quantity is assigned. Zero reaches the range return; blank text and an oversized integer reach the catch. None of those paths assigns the field. The handler then displays the returned message. `fieldLabel.setLabelFor(input)` associates the visible label with the field. Both `save.setOnAction(submit)` and `input.setOnAction(submit)` register the same handler. The complete getter check below reports stored quantity after every request; native form actions separately verify the visible messages and input paths.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

Expected output:

```text
Saved quantity: 0
```

Common error: Treating the field's starting text as a value already saved. Treating the earlier notebook output as a live model display.

</details>


### Check the saved model after every request

Predict each returned message and stored quantity for the requests in this complete ordinary Java check. Run it after writing your trace, then compare each getter result with your prediction. This program creates its own QuantityModel and does not open a window. It tests the same model rules used by the form; native input and status observations remain separate evidence. Explain why both kinds of check are needed.

My prediction:

My actual field text and status observations:

My model getter evidence:

My explanation using the relevant source:

What I changed after comparing the evidence:


In [ ]:
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
QuantityModel model = new QuantityModel();
String[] requests = {"4", "0", "6", "", "six", "2147483648", " 3 ", "1", "5", "word", "2"};
for (String request : requests) {
    System.out.println(model.save(request));
    System.out.println("Stored quantity: " + model.getQuantity());
}


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

This uses the exact QuantityModel body from the form in a fresh model. Each request prints the returned message and then getQuantity. The stored value stays four through both range errors, blank input, a word and integer overflow. Padded three and both valid endpoints succeed. A later rejected word leaves five saved; the final two proves recovery. No window is created by this separate model check.

```java
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
QuantityModel model = new QuantityModel();
String[] requests = {"4", "0", "6", "", "six", "2147483648", " 3 ", "1", "5", "word", "2"};
for (String request : requests) {
    System.out.println(model.save(request));
    System.out.println("Stored quantity: " + model.getQuantity());
}
```

Expected output:

```text
Saved: 4
Stored quantity: 4
Use a quantity from 1 to 5.
Stored quantity: 4
Use a quantity from 1 to 5.
Stored quantity: 4
Enter a whole number from 1 to 5.
Stored quantity: 4
Enter a whole number from 1 to 5.
Stored quantity: 4
Enter a whole number from 1 to 5.
Stored quantity: 4
Saved: 3
Stored quantity: 3
Saved: 1
Stored quantity: 1
Saved: 5
Stored quantity: 5
Enter a whole number from 1 to 5.
Stored quantity: 5
Saved: 2
Stored quantity: 2
```

Common error: Treating the initial zero marker as an accepted submission. Replacing the model between requests and losing the preservation test.

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete validation and shared submission

Copy the displayed incomplete program into the empty work cell and replace all five markers. Reject candidates outside 1 through 5 before saving, connect the visible label to its field, and make button and field Enter use the same handler. Explain why the saved-state assignment belongs after the rejecting return. Run only after completing the markers, then repeat the native submission and cleanup checks.

This sample is for repair:

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (__RANGE_CHECK__) {
                return "Use a quantity from 1 to 5.";
            }
            __SAVE_VALUE__
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    __ASSOCIATE_LABEL__
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = __SUBMIT_BODY__;
    save.setOnAction(submit);
    __ENTER_HANDLER__
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```


My prediction:

My actual field text and status observations:

My model getter evidence:

My explanation using the relevant source:

What I changed after comparing the evidence:

<details>
<summary>Show answer</summary>

Use the two range comparisons joined by ||. The rejecting branch returns before `quantity = candidate;`, so invalid candidates cannot replace saved state. Associate the label with input. The submit lambda reads the current field text, calls the model and updates status. Register that same handler for Enter in the field as well as the already supplied button registration.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

Expected output:

```text
Saved quantity: 0
```

Common error: Joining the outside-range comparisons with &&. Copying a stale field value before the action occurs. Constructing different validation behavior for the two input paths.

</details>


### Change the starting text without saving it

Keep the original program, then change only the TextField starting String from `4` to ` 3 `, with one surrounding space on each side. Predict the initial output and status before running. Submit the starting text with Enter, submit word with the button, then submit 2 with Enter. Compare what the field contains with what the model accepts. Keep your prediction and explain the effect of trim. Close the window with Alt+F4, rerun the complete modified program to check its starting text and status again, then close it.


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});


My prediction:

My actual field text and status observations:

My model getter evidence:

My explanation using the relevant source:

What I changed after comparing the evidence:

<details>
<summary>Show answer</summary>

Only the field's starting String changes. The fresh model still reports Saved quantity: 0 and status still says Nothing saved. Submitting the padded three displays Saved: 3 because trim removes its surrounding spaces before parsing. The word displays Enter a whole number from 1 to 5. without replacing three. The final 2 displays Saved: 2. The field text is a proposed value, so merely displaying padded three does not save it.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField(" 3 ");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

Expected output:

```text
Saved quantity: 0
```

Common error: Changing the model constructor as well as the field text. Removing the spaces from the test before it reaches the TextField.

</details>


### Repair Enter submission

The displayed complete diagnostic differs from the successful Supply Request in one registration. Predict whether Enter in the field and a pointer click on Save quantity will agree. Run this diagnostic in a separate cell when ready to inspect its window. Perform four submissions in order: (1) focus the field containing 4 and press Enter; (2) click Save quantity; (3) select the field text, replace it with 3, keep focus there and press Enter; (4) click Save quantity again. Record status after every submission. Close the diagnostic with Alt+F4. Repair the complete program in the empty work cell, run it, and repeat the input sequence and input methods from the earlier run task to check the repaired form. Close the repaired window, rerun its complete program to check fresh initial state, and close it again. Explain why changing the range check would not repair this input-path problem.

This sample is for repair:

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```


My prediction:

My actual field text and status observations:

My model getter evidence:

My explanation using the relevant source:

What I changed after comparing the evidence:

<details>
<summary>Show answer</summary>

The diagnostic has no action handler on the TextField. Enter on its initial 4 leaves Nothing saved.; clicking Save quantity displays Saved: 4. After editing the field to 3, Enter leaves Saved: 4; clicking the button displays Saved: 3. The model's range rule is unchanged. Restore `input.setOnAction(submit);` so Enter and the button use the same submit behavior. In the complete repaired program, Enter submits current field text and updates status, just as the pointer path does.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

Expected output:

```text
Saved quantity: 0
```

Common error: Changing QuantityModel.save even though the pointer path already uses the required rules. Checking the console construction line to decide whether Enter submitted.

</details>


## Independent Practice

### Build the Loan Form

Create a LoanModel and Loan Form using a TextField initially 3, a Loan days (1 to 7) label associated with that field, a Save loan button, and visible status. Store only whole numbers from 1 through 7 after trimming whitespace. Return Saved: n on success, Use 1 to 7 days. for an out-of-range number, and Enter a whole number from 1 to 7. for nonnumeric input. Keep the previous saved days on any rejection. Use one `EventHandler<ActionEvent>` for button and Enter. Test 3, 1, 7, 0, 8, blank text, a word, an oversized integer, surrounding spaces and a valid submission after an error. Include all imports and fresh model/control objects in the complete program, using the supplied Fx setup for UI work. Start status with Nothing saved., use a 420 by 280 Scene, VBox gap 12, padding 20, and wrapping status text. The construction report is Saved days: followed by the model getter. Before opening the solution, write and run your own program, compare native feedback with your planned rules, and explain the label association, assignment placement and shared handler.

My prediction:

My actual field text and status observations:

My model getter evidence:

My explanation using the relevant source:

What I changed after comparing the evidence:


### Test saved state and visible recovery

Keep one complete Loan Form program. Start a fresh window, then submit these inputs in order without rerunning between them: `3`, `1`, `7`, `0`, `8`, empty text, `word`, `2147483648`, ` 5 `, `word`, and `2`. Alternate pointer and Enter submissions and include focused-button Space activation. Before each action, predict the feedback and saved days. Record the full visible status after each action. Separately run a complete ordinary LoanModel check that calls getDays after each request; an error screenshot alone cannot establish saved state. Explain which inputs fail parsing, which fail the range check, why each rejection preserves the earlier value, and how the final valid request proves recovery. Close the native window, reopen the complete GUI program and confirm fresh initial state, then close it again.

My prediction:

My actual field text and status observations:

My model getter evidence:

My explanation using the relevant source:

What I changed after comparing the evidence:


<details>
<summary>Show answer</summary>

LoanModel starts with zero as the nothing-saved marker. It trims and parses into candidate, checks 1 through 7, and assigns days only on success. Both pointer and Enter invoke one submit handler and display the returned message. The label is explicitly associated with input. The complete source below matches the required Loan Form. Test it with native actions, then use the separate ordinary getter matrix to verify that each rejected request leaves the last valid days unchanged.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class LoanModel {
    private int days;
    public LoanModel() { days = 0; }
    public int getDays() { return days; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 7) {
                return "Use 1 to 7 days.";
            }
            days = candidate;
            return "Saved: " + days;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 7.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    LoanModel model = new LoanModel();
    Label fieldLabel = new Label("Loan days (1 to 7)");
    TextField input = new TextField("3");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save loan");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Loan Form");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved days: " + model.getDays());
});
```

Expected output:

```text
Saved days: 0
```

Common error: Accepting an invalid value by assigning days before the check. Using the quantity range instead of the loan range. Providing a visible label without its association to the TextField.

**Additional test: loan_model_matrix.** This separate complete program uses the exact LoanModel body and prints getDays after every save attempt. Successful 3, 1 and 7 establish each valid value. The range errors 0 and 8, blank text, word and oversized integer preserve seven. Padded five succeeds; the next word preserves five; the final two succeeds. The output proves saved-state behavior for this model sequence. The native form test separately verifies control association, field editing, pointer/keyboard paths and visible feedback.

```java
class LoanModel {
    private int days;
    public LoanModel() { days = 0; }
    public int getDays() { return days; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 7) {
                return "Use 1 to 7 days.";
            }
            days = candidate;
            return "Saved: " + days;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 7.";
        }
    }
}
LoanModel model = new LoanModel();
String[] requests = {"3", "1", "7", "0", "8", "", "word", "2147483648", " 5 ", "word", "2"};
for (String request : requests) {
    System.out.println(model.save(request));
    System.out.println("Stored days: " + model.getDays());
}
```

Expected output:

```text
Saved: 3
Stored days: 3
Saved: 1
Stored days: 1
Saved: 7
Stored days: 7
Use 1 to 7 days.
Stored days: 7
Use 1 to 7 days.
Stored days: 7
Enter a whole number from 1 to 7.
Stored days: 7
Enter a whole number from 1 to 7.
Stored days: 7
Enter a whole number from 1 to 7.
Stored days: 7
Saved: 5
Stored days: 5
Enter a whole number from 1 to 7.
Stored days: 5
Saved: 2
Stored days: 2
```

</details>


## Summary

A TextField holds proposed text. Submission reads its current value. Parse into a candidate and validate the range before assigning saved state. Show a clear message, associate the visible label with its field, and use one handler for button and Enter submission.

Close the answers. Explain why field text, status text, and saved model state can differ after an invalid request. Trace a valid save, rejection, and recovery.


## Reflection

Design a small form for a campus request with a numeric range. State its initial marker, valid range, two distinct rejection messages, and keyboard submission behavior. Describe a test that proves an invalid edit preserves the last valid model value.

**My design and explanation:**

Module 13 introduces background work and coordination. You will keep slow or waiting operations away from the JavaFX Application Thread while reporting their results through the interface.


## Supplemental Reading

- [JavaFX 21 TextField API](https://openjfx.io/javadoc/21/javafx.controls/javafx/scene/control/TextField.html) documents editable text and action events.
- [JavaFX 21 Label API](https://openjfx.io/javadoc/21/javafx.controls/javafx/scene/control/Label.html) documents labelFor association.
- [JavaFX 21 EventHandler API](https://openjfx.io/javadoc/21/javafx.base/javafx/event/EventHandler.html) defines the shared callback contract.
- [Java 21 Integer API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Integer.html#parseInt(java.lang.String)) documents integer parsing and parse failures.
- [Java 21 String API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/String.html#trim()) describes removal of surrounding characters through trim.
